# Flood Detection v4

In [1]:
%%capture
# ── 1. INSTALL DEPENDENCIES ──────────────────────────────────────────────────
# Save numpy version FIRST
import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"], capture_output=True, text=True)
NP_VER = r.stdout.strip()

# Install terratorch WITH all its dependencies
!pip install -q terratorch segmentation-models-pytorch albumentations

# Force numpy back to what Kaggle originally had
!pip install -q --force-reinstall --no-deps numpy=={NP_VER}

In [2]:
# ── 2. IMPORTS & SEEDS ────────────────────────────────────────────────────────
import os, random, warnings, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
import lightning.pytorch as pl
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from scipy import ndimage
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger
from torchmetrics import JaccardIndex, Accuracy
import segmentation_models_pytorch as smp
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Reproducibility — Rule 7.2
SEED = 42
pl.seed_everything(SEED, workers=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Patch Lightning checkpoint loading for PyTorch 2.6+
# (our own checkpoints are trusted, produced during this run)
import lightning.fabric.utilities.cloud_io as _cloud_io
_original_load = _cloud_io._load
def _patched_load(path_or_url, map_location=None, weights_only=True):
    return _original_load(path_or_url, map_location=map_location, weights_only=False)
_cloud_io._load = _patched_load

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

Seed set to 42


PyTorch: 2.10.0+cu128
NumPy: 2.0.2
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# ── 3. PATHS & CONFIG ─────────────────────────────────────────────────────────
DATA_DIR       = Path("/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data")
IMAGE_DIR      = DATA_DIR / "image"
LABEL_DIR      = DATA_DIR / "label"
SPLIT_DIR      = DATA_DIR / "split"
TEST_IMAGE_DIR = DATA_DIR / "prediction" / "image"

OUT_DIR  = Path("/kaggle/working")
CKPT_DIR = OUT_DIR / "checkpoints"
PRED_DIR = OUT_DIR / "prediction"
LOG_DIR  = OUT_DIR / "logs"
for d in [CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Model config
NUM_CLASSES  = 3
IN_CHANNELS  = 6     # SAR-HH, SAR-HV, Green, Red, NIR, SWIR
IMG_SIZE     = 512
IGNORE_INDEX = -1

# Prithvi config — weights downloaded programmatically from HuggingFace (Rule 6.2.3)
BACKBONE     = "prithvi_eo_v2_300_tl"
BANDS        = [1, 2, 3, 4, 5, 6]
NUM_FRAMES   = 1
DECODER      = "UperNetDecoder"
DECODER_CH   = 256

# Training config
PRITHVI_FROZEN_EPOCHS   = 5
PRITHVI_UNFROZEN_EPOCHS = 25
SMP_EPOCHS              = 40

PRITHVI_BATCH  = 2
PRITHVI_ACCUM  = 8    # effective batch = 16
SMP_BATCH      = 8

CLASS_WEIGHTS = [1.0, 4.0, 3.0]   # placeholder, computed from data below

print("Config ready.")
print(f"Train split: {SPLIT_DIR / 'train.txt'}")
print(f"Pred images: {TEST_IMAGE_DIR}")

Config ready.
Train split: /kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/train.txt
Pred images: /kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/prediction/image


In [4]:
# ── 4. COMPUTE CLASS WEIGHTS FROM TRAINING DATA ──────────────────────────────
def compute_class_weights():
    with open(SPLIT_DIR / "train.txt") as f:
        stems = [l.strip() for l in f if l.strip()]
    counts = np.zeros(3, dtype=np.int64)
    for stem in stems:
        p = LABEL_DIR / f"{stem}_label.tif"
        if p.exists():
            with rasterio.open(p) as src:
                lbl = src.read(1).astype(np.int64)
            for c in range(3):
                counts[c] += (lbl == c).sum()
    total = counts.sum()
    freqs = counts / (total + 1e-9)
    for i, name in enumerate(["No Flood", "Flood", "Water Body"]):
        print(f"  Class {i} ({name}): {freqs[i]*100:.2f}%  ({counts[i]:,} px)")
    raw_w = 1.0 / (freqs + 1e-6)
    raw_w = raw_w / raw_w.min()
    raw_w = np.clip(raw_w, 1.0, 5.0)
    print("Class weights:", [round(w, 3) for w in raw_w.tolist()])
    return raw_w.tolist()

CLASS_WEIGHTS = compute_class_weights()

  Class 0 (No Flood): 65.25%  (10,091,613 px)
  Class 1 (Flood): 15.25%  (2,358,596 px)
  Class 2 (Water Body): 19.50%  (3,016,287 px)
Class weights: [1.0, 4.279, 3.346]


In [5]:
# ── 5. DATASET ────────────────────────────────────────────────────────────────
def load_image(path):
    """Load TIF, strip label band if 7 channels, return (6, H, W) float32."""
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)
    if img.shape[0] == 7:
        img = img[1:]  # strip label band
    return img  # (6, H, W)


train_tfms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30,
                       border_mode=0, p=0.4),
    A.GaussNoise(var_limit=(0.001, 0.01), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
    A.Normalize(),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])


class FloodDataset(Dataset):
    def __init__(self, stems, image_dir, label_dir=None,
                 transforms=None, is_test=False):
        self.stems     = stems
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.tfms      = transforms
        self.is_test   = is_test

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img  = load_image(self.image_dir / f"{stem}_image.tif")  # (6, H, W)
        img  = img.transpose(1, 2, 0)  # (H, W, 6) for albumentations

        if self.is_test:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.int64)
        else:
            with rasterio.open(self.label_dir / f"{stem}_label.tif") as src:
                mask = src.read(1).astype(np.int64)

        if self.tfms:
            aug  = self.tfms(image=img, mask=mask)
            img  = aug["image"]   # (6, H, W) tensor
            mask = aug["mask"]    # (H, W) tensor

        return img, mask.long(), stem


# Load splits
def load_stems(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_stems = load_stems(SPLIT_DIR / "train.txt")
val_stems   = load_stems(SPLIT_DIR / "val.txt")
pred_stems  = load_stems(SPLIT_DIR / "pred.txt")

train_ds = FloodDataset(train_stems, IMAGE_DIR, LABEL_DIR, train_tfms)
val_ds   = FloodDataset(val_stems,   IMAGE_DIR, LABEL_DIR, val_tfms)
pred_ds  = FloodDataset(pred_stems,  TEST_IMAGE_DIR, transforms=val_tfms, is_test=True)

train_smp_loader = DataLoader(train_ds, batch_size=SMP_BATCH, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True)
val_smp_loader   = DataLoader(val_ds,   batch_size=SMP_BATCH, shuffle=False,
                               num_workers=2, pin_memory=True)
pred_loader      = DataLoader(pred_ds,  batch_size=1, shuffle=False,
                               num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Pred: {len(pred_ds)}")

Train: 59 | Val: 10 | Pred: 19


In [6]:
# ── 6. LOSS FUNCTION ──────────────────────────────────────────────────────────
dice_loss_fn = smp.losses.DiceLoss(mode="multiclass")
ce_weights   = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(DEVICE)
ce_loss_fn   = nn.CrossEntropyLoss(weight=ce_weights)

def combined_loss(pred, target):
    return dice_loss_fn(pred, target) + ce_loss_fn(pred, target)

print("Loss: Dice + weighted CrossEntropy")
print("Class weights:", CLASS_WEIGHTS)

Loss: Dice + weighted CrossEntropy
Class weights: [1.0, 4.278631139484929, 3.345695127922405]


In [7]:
# ── 7. PRITHVI MODEL (Lightning Module) ───────────────────────────────────────
# Prithvi pretrained weights are downloaded from HuggingFace (Rule 6.2.3 compliant)
from terratorch.tasks import SemanticSegmentationTask


class PrithviFloodModel(pl.LightningModule):
    """
    Prithvi EO v2 300M-TL with UperNet decoder.
    Direct 6-channel input — no projection layer.
    Pretrained weights downloaded programmatically from HuggingFace.
    """
    def __init__(self, lr=2e-5, weight_decay=0.05, freeze_backbone=False):
        super().__init__()
        self.save_hyperparameters()

        necks = [dict(name="ReshapeTokensToImage", effective_time_dim=NUM_FRAMES)]
        model_args = dict(
            backbone_pretrained   = True,
            backbone              = BACKBONE,
            backbone_bands        = BANDS,
            backbone_num_frames   = NUM_FRAMES,
            decoder               = DECODER,
            decoder_channels      = DECODER_CH,
            decoder_scale_modules = True,
            num_classes           = NUM_CLASSES,
            head_dropout          = 0.15,
            necks                 = necks,
            rescale               = True,
        )
        self.task = SemanticSegmentationTask(
            model_args        = model_args,
            plot_on_val       = False,
            class_weights     = CLASS_WEIGHTS,
            loss              = "focal",
            lr                = lr,
            optimizer         = "AdamW",
            optimizer_hparams = dict(weight_decay=weight_decay),
            ignore_index      = IGNORE_INDEX,
            freeze_backbone   = freeze_backbone,
            freeze_decoder    = False,
            model_factory     = "EncoderDecoderFactory",
        )
        self.backbone_model = self.task.model

        # Custom combined loss for our training
        self.dice_loss = smp.losses.DiceLoss(mode="multiclass")
        self.ce_loss   = nn.CrossEntropyLoss(
            weight=torch.tensor(CLASS_WEIGHTS, dtype=torch.float32),
            ignore_index=IGNORE_INDEX)

        kw = dict(task="multiclass", num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)
        self.train_iou  = JaccardIndex(**kw, average="macro")
        self.val_iou    = JaccardIndex(**kw, average="macro")
        self.val_iou_pc = JaccardIndex(**kw, average="none")

    def forward(self, x):
        # x: (B, 6, H, W) from dataloader
        # Prithvi expects (B, C, T, H, W)
        x = x.unsqueeze(2)  # (B, 6, 1, H, W)
        out = self.backbone_model(x)
        logits = out.output if hasattr(out, "output") else out
        if isinstance(logits, dict):
            logits = logits["output"]
        return logits

    def _step(self, batch):
        imgs, lbls, _ = batch
        logits = self(imgs)
        if logits.shape[-2:] != lbls.shape[-2:]:
            logits = F.interpolate(logits, size=lbls.shape[-2:],
                                   mode="bilinear", align_corners=False)
        loss = self.dice_loss(logits, lbls) + self.ce_loss(logits, lbls)
        return loss, logits, lbls

    def training_step(self, batch, _):
        loss, logits, lbls = self._step(batch)
        self.train_iou(logits.argmax(1), lbls)
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train/mIoU", self.train_iou, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        loss, logits, lbls = self._step(batch)
        preds = logits.argmax(1)
        self.val_iou(preds, lbls)
        self.val_iou_pc(preds, lbls)
        self.log("val/loss", loss,          on_epoch=True, prog_bar=True)
        self.log("val/mIoU", self.val_iou,  on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        iou_pc = self.val_iou_pc.compute()
        for i, name in enumerate(["no_flood", "flood", "water_body"]):
            self.log(f"val/IoU_{name}", iou_pc[i], prog_bar=(name == "flood"))
        self.val_iou_pc.reset()

    def configure_optimizers(self):
        backbone_params = list(self.backbone_model.encoder.parameters()) \
                          if hasattr(self.backbone_model, "encoder") else []
        backbone_ids  = {id(p) for p in backbone_params}
        other_params  = [p for p in self.parameters() if id(p) not in backbone_ids]
        groups = [
            {"params": backbone_params, "lr": self.hparams.lr * 0.1},
            {"params": other_params,    "lr": self.hparams.lr},
        ]
        opt = torch.optim.AdamW(groups, weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=PRITHVI_UNFROZEN_EPOCHS, eta_min=1e-7)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}


print("PrithviFloodModel defined.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


PrithviFloodModel defined.


In [8]:
# ── 8. PRITHVI DATAMODULE ─────────────────────────────────────────────────────
class PrithviDM(pl.LightningDataModule):
    def __init__(self):
        super().__init__()

    def setup(self, stage=None):
        pass  # datasets already created in Cell 5

    def train_dataloader(self):
        return DataLoader(train_ds, batch_size=PRITHVI_BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True,
                          persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(val_ds, batch_size=PRITHVI_BATCH, shuffle=False,
                          num_workers=2, pin_memory=True,
                          persistent_workers=True)

prithvi_dm = PrithviDM()
print("PrithviDM ready.")

PrithviDM ready.


In [9]:
# ── 9. TRAIN PRITHVI (2 phases) ───────────────────────────────────────────────

def make_trainer(max_epochs, ckpt_cb_list, log_name, grad_accum=1):
    return pl.Trainer(
        accelerator             = "auto",
        devices                 = "auto",
        precision               = "16-mixed",
        max_epochs              = max_epochs,
        accumulate_grad_batches = grad_accum,
        gradient_clip_val       = 1.0,
        check_val_every_n_epoch = 5,
        log_every_n_steps       = 10,
        logger                  = CSVLogger(str(LOG_DIR), name=log_name),
        callbacks               = ckpt_cb_list + [LearningRateMonitor("epoch")],
        enable_progress_bar     = True,
    )


# Phase A: Frozen backbone warm-up
prithvi = PrithviFloodModel(lr=5e-4, freeze_backbone=True)
ckpt_A  = ModelCheckpoint(
    monitor="val/mIoU", mode="max", dirpath=str(CKPT_DIR),
    filename="prithvi-phA-best", save_top_k=1,
    save_last=False, save_weights_only=True)

print(f"[Prithvi Phase A] {PRITHVI_FROZEN_EPOCHS} epochs | frozen backbone")
make_trainer(PRITHVI_FROZEN_EPOCHS, [ckpt_A], "prithvi_A",
             grad_accum=PRITHVI_ACCUM).fit(prithvi, datamodule=prithvi_dm)
print("Phase A done. Best:", ckpt_A.best_model_path)


# Phase B: Full fine-tune
prithvi_B = PrithviFloodModel.load_from_checkpoint(
    ckpt_A.best_model_path, lr=2e-5, freeze_backbone=False, strict=False)
for p in prithvi_B.parameters():
    p.requires_grad = True

# Free disk — delete Phase A checkpoint after loading
try: os.remove(ckpt_A.best_model_path)
except: pass

ckpt_B = ModelCheckpoint(
    monitor="val/mIoU", mode="max", dirpath=str(CKPT_DIR),
    filename="prithvi-phB-best", save_top_k=1,
    save_last=False, save_weights_only=True)

print(f"[Prithvi Phase B] {PRITHVI_UNFROZEN_EPOCHS} epochs | full fine-tune")
make_trainer(PRITHVI_UNFROZEN_EPOCHS, [ckpt_B], "prithvi_B",
             grad_accum=PRITHVI_ACCUM).fit(prithvi_B, datamodule=prithvi_dm)

BEST_PRITHVI = ckpt_B.best_model_path
print("Prithvi training complete. Best:", BEST_PRITHVI)

2026-04-04 12:37:43,332 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"
2026-04-04 12:37:43,406 - INFO - HTTP Request: GET https://huggingface.co/api/models/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/xet-read-token/63adbd39c271da4c42f447e69b1a7c91a338cdc9 "HTTP/1.1 200 OK"


Prithvi_EO_V2_300M_TL.pt:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/terratorch/models/decoders/upernet_decoder.py:39: UserWarning: DeprecationWarning: scale_modules is deprecated and will be removed in future versions. Use LearnedInterpolateToPyramidal neck instead.
  warnings.warn(
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[Prithvi Phase A] 5 epochs | frozen backbone


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ task           │ SemanticSegmentationTask │  318 M │ train │     0 │
│ 1 │ backbone_model │ PixelWiseModel           │  318 M │ train │     0 │
│ 2 │ dice_loss      │ DiceLoss                 │      0 │ train │     0 │
│ 3 │ ce_loss        │ CrossEntropyLoss         │      0 │ train │     0 │
│ 4 │ train_iou      │ MulticlassJaccardIndex   │      0 │ train │     0 │
│ 5 │ val_iou        │ MulticlassJaccardIndex   │      0 │ train │     0 │
│ 6 │ val_iou_pc     │ MulticlassJaccardIndex   │      0 │ train │     0 │
└───┴────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 M                                                                                                
Total estimated model params size (MB): 1.3 K                                                                      
Modules in train mode: 657                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=25` reached.


Prithvi training complete. Best: /kaggle/working/checkpoints/prithvi-phB-best.ckpt


In [17]:
# ── 10. DEFINE UNet++ & FPN ───────────────────────────────────────────────────
# ImageNet weights downloaded programmatically via smp/timm (Rule 6.2.3 compliant)

unetpp = smp.UnetPlusPlus(
    encoder_name    = "efficientnet-b4",
    encoder_weights = "imagenet",
    in_channels     = IN_CHANNELS,
    classes         = NUM_CLASSES,
    activation      = None,
).to(DEVICE)

fpn = smp.FPN(
    encoder_name    = "resnet50",
    encoder_weights = "imagenet",
    in_channels     = IN_CHANNELS,
    classes         = NUM_CLASSES,
    activation      = None,
).to(DEVICE)

print(f"UNet++ params: {sum(p.numel() for p in unetpp.parameters()) / 1e6:.1f}M")
print(f"FPN params:    {sum(p.numel() for p in fpn.parameters()) / 1e6:.1f}M")

UNet++ params: 20.8M
FPN params:    26.1M


In [19]:
# ── 11. TRAIN UNet++ & FPN (memory-optimized) ────────────────────────────────
import gc

# Force cleanup
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

SMP_BATCH = 4  # reduced from 8 to fit in memory
scaler = torch.amp.GradScaler("cuda")  # mixed precision

# Recreate dataloaders with smaller batch
train_smp_loader = DataLoader(train_ds, batch_size=SMP_BATCH, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True)
val_smp_loader   = DataLoader(val_ds,   batch_size=SMP_BATCH, shuffle=False,
                               num_workers=2, pin_memory=True)


def train_epoch_amp(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss = 0
    for imgs, masks, _ in loader:
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            preds = model(imgs)
            loss  = combined_loss(preds, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    if scheduler:
        scheduler.step()
    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader):
    model.eval()
    iou_metric = JaccardIndex(task="multiclass", num_classes=NUM_CLASSES,
                               average="none").to(DEVICE)
    for imgs, masks, _ in loader:
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        with torch.amp.autocast("cuda"):
            preds = model(imgs).argmax(1)
        iou_metric.update(preds, masks)
    iou_pc = iou_metric.compute().cpu()
    return iou_pc, iou_pc.mean().item()


# Optimizers
opt_unetpp = torch.optim.AdamW(unetpp.parameters(), lr=2e-4, weight_decay=1e-4)
opt_fpn    = torch.optim.AdamW(fpn.parameters(),    lr=2e-4, weight_decay=1e-4)
sched_unetpp = torch.optim.lr_scheduler.CosineAnnealingLR(opt_unetpp, T_max=SMP_EPOCHS, eta_min=1e-6)
sched_fpn    = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fpn,    T_max=SMP_EPOCHS, eta_min=1e-6)

best_unetpp_iou = 0
best_fpn_iou    = 0

for epoch in range(1, SMP_EPOCHS + 1):
    loss_u = train_epoch_amp(unetpp, train_smp_loader, opt_unetpp, sched_unetpp)
    loss_f = train_epoch_amp(fpn,    train_smp_loader, opt_fpn,    sched_fpn)

    if epoch % 5 == 0 or epoch == SMP_EPOCHS:
        iou_u, miou_u = val_epoch(unetpp, val_smp_loader)
        iou_f, miou_f = val_epoch(fpn,    val_smp_loader)
        print(f"Epoch {epoch:3d} | UNet++ loss={loss_u:.4f} mIoU={miou_u:.4f} "
              f"[NF={iou_u[0]:.3f} FL={iou_u[1]:.3f} WB={iou_u[2]:.3f}] | "
              f"FPN loss={loss_f:.4f} mIoU={miou_f:.4f} "
              f"[NF={iou_f[0]:.3f} FL={iou_f[1]:.3f} WB={iou_f[2]:.3f}]")

        if miou_u > best_unetpp_iou:
            best_unetpp_iou = miou_u
            torch.save(unetpp.state_dict(), str(CKPT_DIR / "unetpp_best.pt"))
        if miou_f > best_fpn_iou:
            best_fpn_iou = miou_f
            torch.save(fpn.state_dict(), str(CKPT_DIR / "fpn_best.pt"))
    else:
        print(f"Epoch {epoch:3d} | UNet++ loss={loss_u:.4f} | FPN loss={loss_f:.4f}")

print(f"\nBest UNet++ mIoU: {best_unetpp_iou:.4f}")
print(f"Best FPN mIoU:    {best_fpn_iou:.4f}")


GPU memory: 0.4GB / 15.6GB
Epoch   1 | UNet++ loss=1.9481 | FPN loss=2.8702
Epoch   2 | UNet++ loss=1.8267 | FPN loss=2.1590
Epoch   3 | UNet++ loss=1.7748 | FPN loss=1.8392
Epoch   4 | UNet++ loss=1.7425 | FPN loss=1.8459
Epoch   5 | UNet++ loss=1.7440 mIoU=0.3410 [NF=0.735 FL=0.074 WB=0.214] | FPN loss=1.8149 mIoU=0.3388 [NF=0.662 FL=0.081 WB=0.273]
Epoch   6 | UNet++ loss=1.7047 | FPN loss=1.8641
Epoch   7 | UNet++ loss=1.6833 | FPN loss=1.7927
Epoch   8 | UNet++ loss=1.6221 | FPN loss=1.6604
Epoch   9 | UNet++ loss=1.6733 | FPN loss=1.6160
Epoch  10 | UNet++ loss=1.6360 mIoU=0.4061 [NF=0.776 FL=0.125 WB=0.317] | FPN loss=1.7314 mIoU=0.2820 [NF=0.625 FL=0.088 WB=0.133]
Epoch  11 | UNet++ loss=1.6209 | FPN loss=1.6558
Epoch  12 | UNet++ loss=1.6358 | FPN loss=1.6743
Epoch  13 | UNet++ loss=1.5957 | FPN loss=1.7278
Epoch  14 | UNet++ loss=1.5262 | FPN loss=1.6050
Epoch  15 | UNet++ loss=1.5820 mIoU=0.3796 [NF=0.790 FL=0.148 WB=0.201] | FPN loss=1.6299 mIoU=0.4344 [NF=0.755 FL=0.168 WB

In [20]:
# ── 12. LOAD BEST CHECKPOINTS FOR INFERENCE ───────────────────────────────────

# Load best Prithvi
infer_prithvi = PrithviFloodModel.load_from_checkpoint(
    BEST_PRITHVI, strict=False).to(DEVICE)
infer_prithvi.eval()
print("Prithvi loaded:", BEST_PRITHVI)

# Load best UNet++
unetpp_best = smp.UnetPlusPlus(
    encoder_name="efficientnet-b4", encoder_weights=None,
    in_channels=IN_CHANNELS, classes=NUM_CLASSES, activation=None).to(DEVICE)
unetpp_best.load_state_dict(torch.load(str(CKPT_DIR / "unetpp_best.pt"),
                                        map_location=DEVICE, weights_only=True))
unetpp_best.eval()
print("UNet++ loaded.")

# Load best FPN
fpn_best = smp.FPN(
    encoder_name="resnet50", encoder_weights=None,
    in_channels=IN_CHANNELS, classes=NUM_CLASSES, activation=None).to(DEVICE)
fpn_best.load_state_dict(torch.load(str(CKPT_DIR / "fpn_best.pt"),
                                     map_location=DEVICE, weights_only=True))
fpn_best.eval()
print("FPN loaded.")
print("All 3 models ready for inference.")

2026-04-04 13:10:46,252 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"


Prithvi loaded: /kaggle/working/checkpoints/prithvi-phB-best.ckpt
UNet++ loaded.
FPN loaded.
All 3 models ready for inference.


In [21]:
# ── 13. ENSEMBLE INFERENCE WITH 4-WAY TTA ─────────────────────────────────────

def tta_predict(model, img):
    """4-way TTA: original + hflip + vflip + both.
    img: (1, 6, H, W) tensor on device.
    Returns: (1, 3, H, W) averaged softmax probabilities.
    """
    augments = [
        (lambda x: x,                       lambda x: x),
        (lambda x: torch.flip(x, [-1]),      lambda x: torch.flip(x, [-1])),
        (lambda x: torch.flip(x, [-2]),      lambda x: torch.flip(x, [-2])),
        (lambda x: torch.flip(x, [-1, -2]),  lambda x: torch.flip(x, [-1, -2])),
    ]
    prob_sum = None
    for aug_fn, deaug_fn in augments:
        x = aug_fn(img)
        with torch.no_grad():
            logits = model(x)
            if isinstance(logits, dict):
                logits = logits["output"]
        if logits.shape[-2:] != img.shape[-2:]:
            logits = F.interpolate(logits, size=img.shape[-2:],
                                   mode="bilinear", align_corners=False)
        probs = torch.softmax(logits, dim=1)
        probs = deaug_fn(probs)
        prob_sum = probs if prob_sum is None else prob_sum + probs
    return prob_sum / len(augments)


def minimal_cleanup(mask, min_blob=50):
    """Remove tiny isolated blobs from flood class. No NDWI disambiguation."""
    cleaned = mask.copy()
    flood_bin = (cleaned == 1).astype(np.uint8)
    if flood_bin.sum() == 0:
        return cleaned
    labeled, n = ndimage.label(flood_bin)
    for i in range(1, n + 1):
        if (labeled == i).sum() < min_blob:
            flood_bin[labeled == i] = 0
    cleaned[mask == 1] = 0
    cleaned[flood_bin == 1] = 1
    return cleaned


# Clear prediction dir
if PRED_DIR.exists():
    shutil.rmtree(PRED_DIR)
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Ensemble weights
W_PRITHVI = 0.50
W_UNETPP  = 0.35
W_FPN     = 0.15

print(f"Ensemble: Prithvi={W_PRITHVI} | UNet++={W_UNETPP} | FPN={W_FPN}")
print(f"Predicting {len(pred_ds)} images with 4-way TTA...\n")

for imgs, _, stems in tqdm(pred_loader):
    stem = stems[0]
    imgs = imgs.to(DEVICE)

    # Get TTA probabilities from each model
    p_prithvi = tta_predict(infer_prithvi, imgs)
    p_unetpp  = tta_predict(unetpp_best,  imgs)
    p_fpn     = tta_predict(fpn_best,     imgs)

    # Weighted softmax ensemble
    ensemble = W_PRITHVI * p_prithvi + W_UNETPP * p_unetpp + W_FPN * p_fpn
    pred_mask = ensemble.argmax(1).squeeze(0).cpu().numpy().astype(np.int16)

    # Minimal cleanup only (NO NDWI postprocessing)
    pred_mask = minimal_cleanup(pred_mask)

    # Save prediction TIF with original georeference
    src_path = TEST_IMAGE_DIR / f"{stem}_image.tif"
    with rasterio.open(src_path) as src:
        meta = src.meta.copy()
    meta.update({"count": 1, "dtype": "int16", "nodata": -1, "compress": "lzw"})

    out_path = PRED_DIR / f"{stem}_image.tif"
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(pred_mask, 1)

    pct_flood = (pred_mask == 1).mean() * 100
    pct_water = (pred_mask == 2).mean() * 100
    print(f"  {stem} | flood={pct_flood:.1f}%  water={pct_water:.1f}%")

print(f"\nInference complete. {len(list(PRED_DIR.glob('*.tif')))} files saved.")
torch.cuda.empty_cache()

Ensemble: Prithvi=0.5 | UNet++=0.35 | FPN=0.15
Predicting 19 images with 4-way TTA...



  5%|▌         | 1/19 [00:02<00:40,  2.22s/it]

  20240529_EO4_RES2_fl_pid_080 | flood=0.6%  water=3.0%


 11%|█         | 2/19 [00:03<00:31,  1.88s/it]

  20240529_EO4_RES2_fl_pid_081 | flood=2.9%  water=8.8%


...

  20240529_EO4_RES2_fl_pid_096 | flood=0.0%  water=0.0%


 95%|█████████▍| 18/19 [00:31<00:01,  1.77s/it]

  20240529_EO4_RES2_fl_pid_097 | flood=1.5%  water=1.1%


100%|██████████| 19/19 [00:33<00:00,  1.76s/it]

  20240529_EO4_RES2_fl_pid_098 | flood=3.8%  water=0.3%

Inference complete. 19 files saved.


In [22]:
# ── 14. GENERATE SUBMISSION (pred.txt = 19 images) ───────────────────────────

def mask_to_rle(mask):
    """Column-major RLE (Kaggle convention). Pixel index starts at 1.
    Returns '0 0' for empty masks as required by competition rules."""
    if mask.sum() == 0:
        return "0 0"
    pixels = mask.flatten(order="F")  # column-major (top-to-bottom, left-to-right)
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


# Build submission from pred.txt stems only (exactly 19 rows)
rows = []
for stem in pred_stems:
    tif_path = PRED_DIR / f"{stem}_image.tif"
    if not tif_path.exists():
        print(f"WARNING: Missing {stem}, using empty mask")
        rows.append({"id": stem, "rle_mask": "0 0"})
        continue
    with rasterio.open(tif_path) as src:
        pred = src.read(1)
    flood_mask = (pred == 1).astype(np.uint8)  # RLE encodes flood class only
    rows.append({"id": stem, "rle_mask": mask_to_rle(flood_mask)})

submission = pd.DataFrame(rows)

# Safety: ensure no blank rle_mask values (competition requires '0 0' not blank)
bad = submission["rle_mask"].isna() | (submission["rle_mask"] == "")
if bad.any():
    submission.loc[bad, "rle_mask"] = "0 0"
    print(f"Fixed {bad.sum()} blank rle_mask entries")

out_csv = OUT_DIR / "submission.csv"
submission.to_csv(out_csv, index=False)
print(f"\nSubmission saved: {out_csv}")
print(f"Total rows:  {len(submission)}")
print(f"Non-empty:   {(submission['rle_mask'] != '0 0').sum()}")
print()
print(submission.to_string(index=False))


Submission saved: /kaggle/working/submission.csv
Total rows:  19
Non-empty:   15

                          id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [23]:
# ── 15. VALIDATE SUBMISSION ───────────────────────────────────────────────────
print("=" * 60)
print("SUBMISSION VALIDATION")
print("=" * 60)

# 1. Check correct number of rows
assert len(submission) == len(pred_stems), \
    f"Expected {len(pred_stems)} rows, got {len(submission)}"
print(f"[PASS] Row count: {len(submission)} (expected {len(pred_stems)})")

# 2. Check all IDs present, no extras
submitted_ids = set(submission["id"].tolist())
expected_ids  = set(pred_stems)
missing = expected_ids - submitted_ids
extra   = submitted_ids - expected_ids
print(f"[{'PASS' if not missing else 'FAIL'}] Missing IDs: {len(missing)}")
print(f"[{'PASS' if not extra else 'FAIL'}] Extra IDs: {len(extra)}")

# 3. Validate RLE format (pairs of integers)
def valid_rle(s):
    if not s or pd.isna(s): return False
    toks = s.split()
    return len(toks) % 2 == 0 and all(t.isdigit() for t in toks)

bad_rle = submission[~submission["rle_mask"].apply(valid_rle)]
print(f"[{'PASS' if len(bad_rle) == 0 else 'FAIL'}] Invalid RLE: {len(bad_rle)}")

# 4. Check no blank/null values (competition rule)
has_blanks = submission["rle_mask"].isna().any() or (submission["rle_mask"] == "").any()
print(f"[{'PASS' if not has_blanks else 'FAIL'}] No blank rle_mask values")

print()
if not missing and not extra and len(bad_rle) == 0 and not has_blanks:
    print("ALL CHECKS PASSED — submission is ready for upload!")
else:
    print("SOME CHECKS FAILED — review above.")

print()
print("Download submission.csv from the Output tab on the right sidebar.")

# Clickable download link
from IPython.display import FileLink
FileLink('/kaggle/working/submission.csv')

SUBMISSION VALIDATION
[PASS] Row count: 19 (expected 19)
[PASS] Missing IDs: 0
[PASS] Extra IDs: 0
[PASS] Invalid RLE: 0
[PASS] No blank rle_mask values

ALL CHECKS PASSED — submission is ready for upload!

Download submission.csv from the Output tab on the right sidebar.


/kaggle/working/submission.csv